# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kabin-ux/fly-rank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os
import subprocess
from pathlib import Path

# Colab setup: clone repo if not present
if Path("/content").exists() and not Path("/content/data/raw/content_refresh_anonymized.csv").exists():
    os.chdir("/content")
    subprocess.run(["git", "clone", "https://github.com/kabin-ux/fly-rank-ml-internship-starter", "."], check=True)


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

---

### Two findings, checked against our own data

**Finding #1 -- "The Anatomy of Growing Content" (tagged CONFIRMED in the paper).**
Where the label comes from: `trend_direction` (up/down) is computed from `trend_pct`, the
30-day-vs-previous-30-day impression change -- the exact same construct behind our own
`is_declining_label`. The paper's finding is a same-snapshot comparison of two `trend_direction`
cohorts: growing pages are reported 37.6% longer and 20% younger than declining pages
(n=74.8K vs 45.6K).

Does the validation design carry the claim? It is descriptive statistics on a large sample, not a
predictive model -- there is no train/test split because nothing is being predicted, so
"validation" in the holdout sense does not directly apply. The real risk is causal direction and
portfolio-dependence: age/length gaps could just as easily be a *consequence* of decline (nobody
invests in a page that is already sliding) as a *cause* of it, and the paper itself calls this "an
observational comparison." The code cell below re-runs the identical group-by on our own 30K-row
slice and gets the **opposite sign**: our "down" cohort is longer and younger than our "up" cohort.
Same label logic, same methodology, a different portfolio slice, the opposite direction. That is
not a contradiction of the paper's own data -- it is a demonstration that a same-snapshot cohort
comparison with no holdout is fragile enough to flip sign across two cuts of the same kind of
data, which argues for a footnote ("directional, portfolio-specific") rather than a bare
"confirmed" tag.

**Finding -- ML Appendix "What Predicts Health?"**
Where the label comes from: `health_score` = impressions (30 pts) + position (30 pts) + CTR
(20 pts) + scroll depth (20 pts) -- a formula the paper states explicitly a few pages earlier. The
Random Forest built to explain that same health score then reports Average Position (43%) and
Impressions (32%) as its top two predictors.

Does the validation design carry the claim? The paper does the honest thing and self-flags this
("the target itself is partly constructed from some of these inputs, so importance is descriptive
rather than causal") -- but an 80/20 holdout split cannot fix it, because the leaky relationship is
structural: it exists identically in train and test, so holdout accuracy stays inflated no matter
how the split is drawn. The code cell below rebuilds a toy version of the same health-score formula
from our own columns and runs the same test: with `avg_position` and `impressions_90d` in the
feature set, R^2 = 0.984 and those two features alone carry about 80% of the importance; drop them
and R^2 collapses, with `content_age_days` and `word_count` (genuinely independent of the formula)
taking over as the top signals. That reproduces leakage-taxonomy pattern #1 (label-derived
features) on our own data -- it confirms the paper's own caveat is doing real work, and it argues
for dropping position/impressions from that appendix chart entirely rather than caveating around
them.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Check two paper findings against our own data

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')

def _find_starter_csv():
    rel = Path("data/raw/content_refresh_anonymized.csv")
    cur = Path.cwd().resolve()
    for _ in range(8):
        cand = cur / rel
        if cand.exists():
            return cand
        if cur.parent == cur:
            break
        cur = cur.parent
    return None

RAW = _find_starter_csv()
assert RAW is not None, f"CSV not found from cwd={Path.cwd().resolve()}"
df = pd.read_csv(RAW)

print("=" * 80)
print("CHECK 1 -- Finding #1 'The Anatomy of Growing Content' (paper: up = longer + younger)")
print("=" * 80)
cohort = df[df['trend_direction'].isin(['up', 'down'])]
summary = cohort.groupby('trend_direction').agg(
    n=('content_id', 'count'),
    avg_word_count=('word_count', 'mean'),
    avg_age_days=('content_age_days', 'mean'),
).round(1)
print(summary)
print("\nPaper: growing pages are longer + younger than declining ones (n=74.8K vs 45.6K).")
print("Our slice: same trend_direction / trend_pct label logic, opposite sign on both columns.")

print("\n" + "=" * 80)
print("CHECK 2 -- ML Appendix 'What Predicts Health?' (label-derived feature reproduction)")
print("=" * 80)
h = df.dropna(subset=['ctr', 'avg_position', 'scroll_rate', 'impressions_90d']).copy()
h['imp_pts'] = (h['impressions_90d'].clip(upper=h['impressions_90d'].quantile(0.95)) /
                h['impressions_90d'].quantile(0.95) * 30)
h['pos_pts'] = (1 - h['avg_position'].clip(upper=50) / 50) * 30
h['ctr_pts'] = h['ctr'].clip(upper=5) / 5 * 20
h['scroll_pts'] = h['scroll_rate'].clip(upper=100) / 100 * 20
h['toy_health'] = h['imp_pts'] + h['pos_pts'] + h['ctr_pts'] + h['scroll_pts']

feature_cols_with = ['avg_position', 'impressions_90d', 'ctr', 'scroll_rate', 'word_count', 'content_age_days']
feature_cols_without = ['word_count', 'content_age_days', 'search_volume', 'competition']

def rf_r2(cols):
    X = h[cols].fillna(0)
    y = h['toy_health']
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=42)
    rf = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
    rf.fit(Xtr, ytr)
    r2 = r2_score(yte, rf.predict(Xte))
    imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
    return r2, imp

r2_with, imp_with = rf_r2(feature_cols_with)
r2_without, imp_without = rf_r2(feature_cols_without)
print(f"R^2 predicting a toy health score WITH avg_position + impressions_90d: {r2_with:.3f}")
print(imp_with.round(3).to_string())
print(f"\nR^2 WITHOUT avg_position + impressions_90d (same toy health score): {r2_without:.3f}")
print(imp_without.round(3).to_string())
print("\nSame pattern the paper's own appendix reports for Average Position (43%) and Impressions (32%):")
print("the label's own ingredients dominate importance and inflate R^2 until they're removed.")


CHECK 1 -- Finding #1 'The Anatomy of Growing Content' (paper: up = longer + younger)
                     n  avg_word_count  avg_age_days
trend_direction                                     
down             16262          3221.8         236.2
up                4388          2998.1         288.5

Paper: growing pages are longer + younger than declining ones (n=74.8K vs 45.6K).
Our slice: same trend_direction / trend_pct label logic, opposite sign on both columns.

CHECK 2 -- ML Appendix 'What Predicts Health?' (label-derived feature reproduction)
R^2 predicting a toy health score WITH avg_position + impressions_90d: 0.984
avg_position        0.475
impressions_90d     0.325
scroll_rate         0.157
ctr                 0.043
word_count          0.000
content_age_days    0.000

R^2 WITHOUT avg_position + impressions_90d (same toy health score): 0.190
content_age_days    0.400
word_count          0.382
search_volume       0.116
competition         0.102

Same pattern the paper's own appe

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

---

### Before (naive random split) vs after (honest client-grouped split)

Same features, same two models as Week 5. Week 5's writeup already intended a client-grouped
split, but that notebook never actually ran a random-split comparison next to it -- and had an
unrunnable gap (`prepare_features` / `df_train` / `df_test` were referenced but never defined in
that notebook's cells). This section rebuilds the pipeline standalone, with the same feature list,
and runs both splits for real.

| Split | n_test | Base rate | LR test AUC | RF test AUC |
|---|---|---|---|---|
| Random 70/30 (before) | 9,000 | 54.2% | 0.696 | 0.760 |
| Client-grouped 70/30 (after) | 10,834 | 55.9% | 0.590 | 0.622 |

The gap is the finding: Random Forest loses 0.139 AUC points and Logistic Regression loses 0.106
the moment test clients are guaranteed to be unseen during training. That gap is memorization the
random split was hiding -- some of what looked like signal was really the model learning
per-client quirks (a client's typical CTR, its content mix) rather than a pattern that
generalizes to a page it has never seen the client of.

Precision@K on the honest grouped split (K=1,708, base rate 55.9%): baseline rule 0.619, Logistic
Regression 0.653, Random Forest 0.679. The model still beats the hand-written baseline under the
split that actually mimics deploying on a brand-new client -- just by a smaller, more honest
margin than the random-split numbers alone would suggest.


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Re-run the Week-5 model under a naive random split vs an honest client-grouped split

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

def _find_starter_csv():
    rel = Path("data/raw/content_refresh_anonymized.csv")
    cur = Path.cwd().resolve()
    for _ in range(8):
        cand = cur / rel
        if cand.exists():
            return cand
        if cur.parent == cur:
            break
        cur = cur.parent
    return None

RAW = _find_starter_csv()
df = pd.read_csv(RAW)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]

def prepare_features(frame, numeric_cols, categorical_cols):
    frame = frame.copy()
    # Missingness follows content_type (data dictionary) -- flag it, don't blind-fillna(0)
    for col in ['search_volume', 'competition', 'cpc']:
        frame[f'has_{col}'] = frame[col].notna().astype(int)
    num = frame[numeric_cols].fillna(0)
    cat = pd.get_dummies(frame[categorical_cols].astype('object'), dummy_na=True)
    flags = frame[[f'has_{c}' for c in ['search_volume', 'competition', 'cpc']]]
    return pd.concat([num, flags, cat], axis=1)

X_all = prepare_features(df, numeric_features, categorical_features)
y_all = df['is_declining_label'].values
groups_all = df['client_id'].values

def fit_eval(X_train, X_test, y_train, y_test, label):
    scaler = StandardScaler()
    Xtr = scaler.fit_transform(X_train)
    Xte = scaler.transform(X_test)
    lr = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1)
    lr.fit(Xtr, y_train)
    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    lr_auc = roc_auc_score(y_test, lr.predict_proba(Xte)[:, 1])
    rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])
    print(f"{label}: n_test={len(y_test)}, base rate={y_test.mean():.1%}, LR AUC={lr_auc:.3f}, RF AUC={rf_auc:.3f}")
    return lr_auc, rf_auc, lr, rf, scaler

print("=" * 80)
print("BEFORE: naive random 70/30 split (ignores that rows repeat within client)")
print("=" * 80)
Xtr_r, Xte_r, ytr_r, yte_r = train_test_split(X_all, y_all, test_size=0.3, random_state=42, stratify=y_all)
random_lr_auc, random_rf_auc, _, _, _ = fit_eval(Xtr_r, Xte_r, ytr_r, yte_r, "Random split")

print("\n" + "=" * 80)
print("AFTER: client-grouped 70/30 split (test clients never appear in train)")
print("=" * 80)
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X_all, y_all, groups=groups_all))
Xtr_g, Xte_g = X_all.iloc[train_idx], X_all.iloc[test_idx]
ytr_g, yte_g = y_all[train_idx], y_all[test_idx]
train_clients = set(df.iloc[train_idx]['client_id'])
test_clients = set(df.iloc[test_idx]['client_id'])
print(f"Train clients: {len(train_clients)}, Test clients: {len(test_clients)}, overlap: {len(train_clients & test_clients)}")
grouped_lr_auc, grouped_rf_auc, lr_g, rf_g, scaler_g = fit_eval(Xtr_g, Xte_g, ytr_g, yte_g, "Grouped split")

print(f"\nAUC drop, random -> grouped: LR {random_lr_auc - grouped_lr_auc:.3f}, RF {random_rf_auc - grouped_rf_auc:.3f}")

# Precision@K on the honest grouped split, baseline included for a fair three-way comparison
def precision_at_k(y_true, y_scores, k):
    if len(y_scores) < k:
        k = len(y_scores)
    idx = np.argsort(-y_scores)[:k]
    return np.mean(y_true[idx])

k_test = int(len(yte_g) * (4732 / len(df)))
baseline_csv = pd.read_csv('work/outputs/baseline_action_score.csv')
df_test_g = df.iloc[test_idx].merge(baseline_csv[['content_id', 'score']], on='content_id', how='left')
baseline_probs = df_test_g['score'].fillna(0).values

lr_pk = precision_at_k(yte_g, lr_g.predict_proba(scaler_g.transform(Xte_g))[:, 1], k_test)
rf_pk = precision_at_k(yte_g, rf_g.predict_proba(Xte_g)[:, 1], k_test)
baseline_pk = precision_at_k(yte_g, baseline_probs, k_test)

print("\n" + "=" * 80)
print(f"PRECISION@K ON THE HONEST (GROUPED) SPLIT -- K={k_test}, base rate={yte_g.mean():.3f}")
print("=" * 80)
result = pd.DataFrame({
    'Method': ['Baseline rule', 'Logistic Regression', 'Random Forest'],
    'Precision@K (grouped split)': [baseline_pk, lr_pk, rf_pk],
})
print(result.to_string(index=False))

# Receipt: regenerated every run (gitignored like the other work/outputs artifacts) --
# the printed numbers above are the ones that go in the writeup.
Path('work/outputs').mkdir(parents=True, exist_ok=True)
pd.DataFrame({
    'split': ['random', 'grouped'],
    'lr_auc': [random_lr_auc, grouped_lr_auc],
    'rf_auc': [random_rf_auc, grouped_rf_auc],
}).to_csv('work/outputs/w06_split_comparison.csv', index=False)
print("\nSaved receipt: work/outputs/w06_split_comparison.csv")


BEFORE: naive random 70/30 split (ignores that rows repeat within client)
Random split: n_test=9000, base rate=54.2%, LR AUC=0.696, RF AUC=0.760

AFTER: client-grouped 70/30 split (test clients never appear in train)
Train clients: 22, Test clients: 10, overlap: 0
Grouped split: n_test=10834, base rate=55.9%, LR AUC=0.590, RF AUC=0.622

AUC drop, random -> grouped: LR 0.106, RF 0.139

PRECISION@K ON THE HONEST (GROUPED) SPLIT -- K=1708, base rate=0.559
             Method  Precision@K (grouped split)
      Baseline rule                     0.618267
Logistic Regression                     0.653396
      Random Forest                     0.679157

Saved receipt: work/outputs/w06_split_comparison.csv


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

---

### Attack checklist, run for real on the final feature set

(22 numeric + 8 categorical columns, same list as Week 5.)

- [x] **No label-derived columns.** Confirmed `trend_direction` and `trend_pct` are absent from
  the feature list (asserted in code, not just eyeballed).
- [x] **No IDs as features.** Confirmed `content_id` and `client_id` are absent -- used only for
  the grouped split.
- [x] **No product flags.** This starter cut ships no FlyRank optimization-flag column (those only
  appear in `work/outputs/baseline_action_score.csv`, which is *our own* rule output, read purely
  as a comparison baseline in Section 2 -- never as a feature). Noting this so the check is not
  silently skipped just because nothing showed up.
- [x] **Window overlap, checked rather than assumed.** `impressions_90d` correlates 0.980 with
  `impressions_last_30d + impressions_prev_30d`, and for a typical row 57% of the 90-day total is
  already inside those two 30-day windows that build the label. That is real structural overlap,
  worth disclosing even before knowing whether it matters.
- [x] **Deliberate leak test (attack your own model).** Added the raw `*_last_30d` / `*_prev_30d`
  columns straight back into the grouped-split feature set. Grouped-split LR AUC jumps from 0.590
  to 0.881 -- an obvious tell that the harness correctly detects leakage when it is really there.
  Those columns stay excluded from the shipped feature set.
- [x] **Milder version of the same test.** Instead of adding columns, dropped the whole `*_90d`
  engagement family (impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d,
  engaged_sessions_90d) that showed the overlap above. Grouped-split LR AUC is 0.606 without them
  vs 0.590 with them -- a ~0.02 difference, i.e. noise. So despite the structural overlap, the
  90-day aggregates are not secretly doing the label's work; the rate features (ctr, avg_position,
  engagement_rate, scroll_rate) carry the real signal. Kept in the feature set, overlap disclosed
  rather than hidden.
- [x] **Base rate printed next to every metric above** (54.2%-55.9% throughout).
- [x] **Metrics recomputed out-of-fold** (held-out test set), never in-sample.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Leakage audit on the final feature set -- same hunt as Week 3, actually run this time

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

def _find_starter_csv():
    rel = Path("data/raw/content_refresh_anonymized.csv")
    cur = Path.cwd().resolve()
    for _ in range(8):
        cand = cur / rel
        if cand.exists():
            return cand
        if cur.parent == cur:
            break
        cur = cur.parent
    return None

RAW = _find_starter_csv()
df = pd.read_csv(RAW)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

numeric_features = [
    'search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
categorical_features = [
    'competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier'
]

def prepare_features(frame, numeric_cols, categorical_cols):
    frame = frame.copy()
    for col in ['search_volume', 'competition', 'cpc']:
        frame[f'has_{col}'] = frame[col].notna().astype(int)
    num = frame[numeric_cols].fillna(0)
    cat = pd.get_dummies(frame[categorical_cols].astype('object'), dummy_na=True)
    flags = frame[[f'has_{c}' for c in ['search_volume', 'competition', 'cpc']]]
    return pd.concat([num, flags, cat], axis=1)

print("=" * 80)
print("CHECKLIST: label-derived columns and IDs")
print("=" * 80)
final_feature_set = numeric_features + categorical_features
for banned in ['trend_direction', 'trend_pct', 'content_id', 'client_id']:
    assert banned not in final_feature_set, f"{banned} leaked into the feature set!"
print("Confirmed absent from the feature set: trend_direction, trend_pct, content_id, client_id.")
print("No FlyRank-style optimization-flag column ships in this starter cut, so there is nothing")
print("product-derived to exclude here -- work/outputs/baseline_action_score.csv is OUR OWN rule")
print("output, read as a comparison baseline in Section 2, never as a feature.")

print("\n" + "=" * 80)
print("WINDOW OVERLAP CHECK: does impressions_90d already contain the label's own windows?")
print("=" * 80)
overlap = df[['impressions_90d', 'impressions_last_30d', 'impressions_prev_30d']].dropna()
overlap['sum_30d_windows'] = overlap['impressions_last_30d'] + overlap['impressions_prev_30d']
corr = overlap['impressions_90d'].corr(overlap['sum_30d_windows'])
share = (overlap['sum_30d_windows'] / overlap['impressions_90d'].replace(0, np.nan)).median()
print(f"corr(impressions_90d, last_30d + prev_30d) = {corr:.3f}")
print(f"median share of impressions_90d already inside those two 30d windows = {share:.2f}")
print("Real structural overlap -- disclosed, then stress-tested below rather than ignored.")

print("\n" + "=" * 80)
print("ATTACK #1: deliberately add the raw 30d windows back in -- does the score jump?")
print("=" * 80)
leak_cols = ['impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d',
             'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
df_leak = df.dropna(subset=leak_cols).copy()
X_leaky = prepare_features(df_leak, numeric_features + leak_cols, categorical_features)
y_leaky = df_leak['is_declining_label'].values
groups_leaky = df_leak['client_id'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr_idx, te_idx = next(gss.split(X_leaky, y_leaky, groups=groups_leaky))
scaler = StandardScaler()
Xtr = scaler.fit_transform(X_leaky.iloc[tr_idx])
Xte = scaler.transform(X_leaky.iloc[te_idx])
lr_leaky = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1).fit(Xtr, y_leaky[tr_idx])
leaky_auc = roc_auc_score(y_leaky[te_idx], lr_leaky.predict_proba(Xte)[:, 1])
print(f"Grouped-split LR AUC WITH the raw last-30d/prev-30d columns added in: {leaky_auc:.3f}")
print("Grouped-split LR AUC on the shipped feature set (Section 2, honest number): 0.590")
print("The jump confirms the harness catches real leakage. These columns stay excluded.")

print("\n" + "=" * 80)
print("ATTACK #2: milder version -- drop the whole *_90d engagement family instead")
print("=" * 80)
cols_90d = ['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d',
            'users_90d', 'engaged_sessions_90d']
numeric_wo_90d = [c for c in numeric_features if c not in cols_90d]
X_wo = prepare_features(df, numeric_wo_90d, categorical_features)
y_all = df['is_declining_label'].values
groups_all = df['client_id'].values
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
tr2, te2 = next(gss2.split(X_wo, y_all, groups=groups_all))
scaler2 = StandardScaler()
Xtr2 = scaler2.fit_transform(X_wo.iloc[tr2])
Xte2 = scaler2.transform(X_wo.iloc[te2])
lr_wo = LogisticRegression(max_iter=1000, random_state=42, n_jobs=-1).fit(Xtr2, y_all[tr2])
wo_auc = roc_auc_score(y_all[te2], lr_wo.predict_proba(Xte2)[:, 1])
print(f"Grouped-split LR AUC WITHOUT any *_90d engagement columns: {wo_auc:.3f}")
print("Grouped-split LR AUC WITH them (the shipped feature set): 0.590")
print("A ~0.02 difference is noise -- despite the structural overlap above, the *_90d aggregates")
print("are not secretly doing the label's work. The rate features (ctr, avg_position,")
print("engagement_rate, scroll_rate) carry the real signal. Kept in the feature set, overlap")
print("disclosed rather than hidden.")


CHECKLIST: label-derived columns and IDs
Confirmed absent from the feature set: trend_direction, trend_pct, content_id, client_id.
No FlyRank-style optimization-flag column ships in this starter cut, so there is nothing
product-derived to exclude here -- work/outputs/baseline_action_score.csv is OUR OWN rule
output, read as a comparison baseline in Section 2, never as a feature.

WINDOW OVERLAP CHECK: does impressions_90d already contain the label's own windows?
corr(impressions_90d, last_30d + prev_30d) = 0.980
median share of impressions_90d already inside those two 30d windows = 0.57
Real structural overlap -- disclosed, then stress-tested below rather than ignored.

ATTACK #1: deliberately add the raw 30d windows back in -- does the score jump?
Grouped-split LR AUC WITH the raw last-30d/prev-30d columns added in: 0.880
Grouped-split LR AUC on the shipped feature set (Section 2, honest number): 0.590
The jump confirms the harness catches real leakage. These columns stay excluded.

A

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

---

### Boldest sentence, before and after

**Original** (Week 5, self-check line):

> "The model must also predict on held-out clients it has never seen -- this is how we know it
> generalizes beyond this snapshot."

That overclaims two ways: "we know it generalizes" states certainty from a single 70/30 client
split on one snapshot, and "beyond this snapshot" implies robustness across time that was never
tested (no time-based split exists anywhere in this project).

**Rewritten:**

> On a client-grouped 70/30 split, Random Forest reached 0.679 precision@K and 0.622 AUC versus a
> 0.619 precision@K rule-based baseline on the same 10 held-out clients (base rate 55.9%) -- a
> directional improvement over the baseline for new clients within this 90-day snapshot. This is
> observed, decision-support evidence for prioritizing which pages a human reviews first; it is
> not evidence the model would hold up on a different time window, since no time-based split has
> been run.

Same underlying result, but the second version names the exact split, the exact numbers, and the
exact thing that was *not* tested -- observed / measured / directional / decision-support, not
"we know it generalizes."


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Receipt for the rewritten claim -- pull the exact numbers from Section 2's saved metrics

import pandas as pd
from pathlib import Path

receipt_path = Path('work/outputs/w06_split_comparison.csv')
assert receipt_path.exists(), "Run Section 2's cell first -- it writes this receipt."
receipt = pd.read_csv(receipt_path)
print("Receipt (regenerated every run, from Section 2):")
print(receipt.to_string(index=False))

print("\nClaim receipt, in full:")
print("  Split: client-grouped 70/30, 10 held-out clients never seen in training")
print("  Base rate on held-out clients: 55.9%")
print("  Random Forest: 0.622 AUC, 0.679 precision@K (K=1708)")
print("  Baseline rule, same held-out clients: 0.619 precision@K")
print("  -> directional improvement over the baseline, for new clients, within this snapshot")
print("  -> NOT tested: a different time window (no time-based split exists in this project)")


Receipt (regenerated every run, from Section 2):
  split   lr_auc   rf_auc
 random 0.696094 0.760348
grouped 0.590362 0.621843

Claim receipt, in full:
  Split: client-grouped 70/30, 10 held-out clients never seen in training
  Base rate on held-out clients: 55.9%
  Random Forest: 0.622 AUC, 0.679 precision@K (K=1708)
  Baseline rule, same held-out clients: 0.619 precision@K
  -> directional improvement over the baseline, for new clients, within this snapshot
  -> NOT tested: a different time window (no time-based split exists in this project)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.